# Persistence algorithm 을 이용한 forecasting 

- Persistence Algorithm : last value 를 next value 로 예측에 사용


- 약간의 noise 를 추가한 seasonality 와 trend 가 있는 인공 합성 time series 이용 

In [ ]:
# !pip install scikit-learn

아래 코드는 다음과 같은 작업을 수행합니다:

trend 함수를 정의하여 주어진 시간(time)에 대한 선형 추세를 계산합니다. 기울기(slope) 인자를 통해 추세의 강도를 조절할 수 있습니다.  

seasonal_pattern 함수를 정의하여 시간(season_time)에 대한 계절성 패턴을 생성합니다. 이 예에서는 코사인 함수와 지수 함수를 사용하여 임의의 패턴을 만듭니다.  

seasonality 함수를 정의하여 시간(time)에 대해 주기적으로 반복되는 계절성 패턴을 생성합니다. 주기(period), 진폭(amplitude), 및 위상(phase) 인자를 사용하여 계절성 패턴의 특성을 조절할 수 있습니다.  

noise 함수를 정의하여 시간(time)에 대한 노이즈를 생성합니다. 노이즈 수준(noise_level)과 시드(seed) 인자를 사용하여 노이즈의 특성을 조절할 수 있습니다.  

In [ ]:
def trend(time, slope=0):
def seasonal_pattern(season_time):
def seasonality(time, period, amplitude=1, phase=0):
def noise(time, noise_level=1, seed=None):
# 기본값, 추세, 계절성 및 노이즈를 결합하여 시계열 데이터(series)를 생성
# noise 추가

### 위에서 생성한 time series 를 train 와 validation set 으로 분할

시간(time) 및 시계열 데이터(series)를 split_time을 기준으로 훈련 세트와 테스트 세트로 나눕니다.  
그래프에서 훈련 세트는 파란색 선으로, 테스트 세트는 주황색 선으로 표시됩니다. 

In [ ]:
# 학습/테스트 데이터 분리 기준점 설정
# 시간 및 시계열 데이터를 학습/테스트로 분리
# 학습/테스트 데이터 시각화

## Naive Forecast

- t-1 시점의 실제값 → t 시점의 예측값으로 사용하는 가장 단순한 예측 방법  
- 다른 모델의 성능 비교를 위한 baseline 으로 주로 활용

 series에서 테스트 세트에 해당하는 부분의 이전 시점의 값을 추출하여 naive_forecast 변수에 저장합니다. 그러면 이전 시점의 값들이 현재 시점의 예측값이 됩니다.

In [ ]:
# series[999:-1] : 인덱스 999(split_time-1)부터 마지막 직전까지 슬라이싱
# t 시점 예측 = t-1 시점의 실제값 (한 칸 앞의 값을 예측값으로 사용)
# 테스트 데이터(series[1000:])와 길이가 동일하게 맞춰짐
# 앞 10개의 naive forecast 값 확인

실제값이 naive_forecast 값과 유사한지 확인해 봅니다.

테스트 세트에 대한 원래 시계열 데이터와 naive forecasting을 그래프로 그립니다.

In [ ]:
# 테스트 구간에서 실제값과 Naive Forecast 비교 시각화

naive forecast 가 one-step behind 인지 그래프상으로 구별 어려우므로 validation period 의 앞부분으로 zoom in.

- naive forecast 가 1 step behind 지연되어 time series 를 그대로 재현하고 있음을 알 수 있습니다.

### naive forecast 와 validation period 값 간의 차이를  mse 와 mae 로 계산

테스트 세트와 naive forecasting 사이의 평균 제곱 오차(MSE)와 평균 절대 오차(MAE)를 계산합니다.

### 위의 값을 모든 prediction model 의 최소 성능의 baseline 으로 한다. 

# Real Data를 이용한 Naive Forecasting

parser 함수는 인자로 문자열 x를 받아서 datetime.strptime 함수를 사용하여 해당 문자열을 '%Y-%m' 형식의 datetime 객체로 변환합니다.  
```
"Month","Sales"
"1-01",266.0
"1-02",145.9
"1-03",183.1
"1-04",119.3
```

In [ ]:
def parser(x):

인터넷에서 샴푸 판매량 데이터셋(CSV 파일)을 읽어들이고, Pandas DataFrame 객체로 변환합니다. 
- parse_dates 매개변수를 사용하여 첫 번째 열(0)을 날짜로 처리하도록 설정합니다.
- date_parser 매개변수를 사용하여 앞서 정의한 parser 함수를 날짜 변환에 사용합니다.
- index_col 매개변수를 사용하여 날짜 열을 DataFrame의 인덱스로 설정합니다.
- header 매개변수를 사용하여 첫 번째 행을 열 이름으로 사용하도록 설정합니다.

In [ ]:
# 샴푸 판매량 데이터셋 URL (GitHub raw 파일)
# CSV 파일 로드 - 첫 번째 컬럼을 인덱스로, 첫 번째 행을 헤더로 설정
# 인덱스 형식이 '1-01' 형태이므로 '190' 을 앞에 붙여 '1901-01' 형태로 변환 후 datetime으로 변환
# 상위 5개 데이터 확인

### Train / Test set 분리

In [ ]:
# 전체 데이터의 60%를 학습, 40%를 테스트로 분리
# 학습/테스트 데이터 시각화

### Persistence Algorithm 정의

이 모델은 이전 시점의 값을 현재 시점의 예측값으로 사용합니다. 다음과 같은 인자를 받습니다:

- series: 시계열 데이터를 포함하는 Pandas Series 또는 DataFrame 객체입니다.
- split_time: 시계열 데이터를 학습 세트와 테스트 세트로 나누는 인덱스입니다.  

함수는 테스트 세트에 해당하는 부분의 이전 시점의 값을 반환합니다.

In [ ]:
def model_persistence(series, split_time):
    # 테스트 시작 직전(split_time-1)부터 마지막 직전까지 슬라이싱
    # → 한 칸 앞의 값을 예측값으로 사용 (t-1 → t 예측)

### 예측 및 평가

In [ ]:
# Naive Forecast 예측값 생성
# 테스트 데이터와 예측값 간의 MSE(평균 제곱 오차) 계산

In [ ]:
# 학습 데이터, 실제값, Naive Forecast 예측값 비교 시각화